# 04 — Faza 1.5: Few-shot RHpE validation z anchor klipów

**Cel:** few-shot V-JEPA-2 cosine similarity per behavior — czy embedding rozróżnia 5 RHpE behaviors:
1. ear_position
2. head_position
3. mouth_open
4. tail_movement
5. eye_expression

**Workflow:**
- **Tryb 1 — Clipping** (Gradio UI): Z `vendor/rhpe_anchored/suggestions.csv` (361 propozycji wygenerowanych przez `tools/subtitle_search.py`) wybierasz 5–10 klipów per behavior, system clipuje przez ffmpeg.
- **Tryb 2 — Evaluation**: V-JEPA-2 embedding extraction, leave-one-out cosine similarity per behavior.

**UWAGA copyright:** Klipy są fragmentami YouTube. Fair use private research. **NIE** commit do publicznego repo. `.gitignore` zawiera `poc/vendor/rhpe_anchored/`.

## 1. Setup

In [ ]:
import os, sys, subprocess, csv, json
from pathlib import Path
from collections import Counter

POC = Path(os.getcwd()).resolve()
if POC.name == 'notebooks':
    POC = POC.parent
os.chdir(POC)

ANCHORED = POC / 'vendor/rhpe_anchored'
SUGGESTIONS_CSV = ANCHORED / 'suggestions.csv'
BEHAVIORS = ['ear_position', 'head_position', 'mouth_open', 'tail_movement', 'eye_expression']
TARGET_PER_BEHAVIOR = 5  # minimum approved klipów per behavior do uruchomienia eval

for b in BEHAVIORS:
    (ANCHORED / b).mkdir(parents=True, exist_ok=True)

with SUGGESTIONS_CSV.open() as f:
    suggestions = list(csv.DictReader(f))
print(f'Suggestions: {len(suggestions)}')
by_b = Counter(s['behavior'] for s in suggestions)
for b in BEHAVIORS:
    n = by_b.get(b, 0)
    n_app = sum(1 for s in suggestions if s['behavior']==b and s['status']=='approved')
    print(f'  {b:20s} pending={n - n_app:>3d}  approved={n_app:>2d}/{TARGET_PER_BEHAVIOR}')

## 2. Tryb 1 — Gradio UI clipping

Uruchom komórkę poniżej. Gradio otworzy UI w nowej karcie (lub inline). Workflow:
1. Wybierz behavior z dropdown
2. UI pokazuje pending propozycje per behavior
3. Kliknij **Play** w video player (Gradio załaduje z timestamp)
4. Jeśli fragment pokazuje konkretny RHpE behavior:
   - **Approve** → ffmpeg tnie clip → zapisuje do `vendor/rhpe_anchored/<behavior>/`
   - **Reject** → kolejny
   - **Edit timestamps** → zmień start/end (sekundy), potem Approve
5. Cel: 5–10 approved per behavior. UI pokazuje counter.

Po osiągnięciu 5 × 5 = 25 klipów → uruchom Tryb 2 (sekcja 3).

In [ ]:
import gradio as gr
import csv as _csv
import re
import shutil

STATE = {'idx': 0, 'behavior': BEHAVIORS[0]}


def reload_suggestions():
    with SUGGESTIONS_CSV.open() as f:
        return list(_csv.DictReader(f))


def save_suggestions(rows):
    if not rows:
        return
    fieldnames = list(rows[0].keys())
    with SUGGESTIONS_CSV.open('w', newline='') as f:
        w = _csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)


def get_pending_for(behavior):
    rows = reload_suggestions()
    pending = [(i, r) for i, r in enumerate(rows) if r['behavior'] == behavior and r['status'] == 'pending']
    return rows, pending


def fmt_status_summary():
    rows = reload_suggestions()
    lines = []
    for b in BEHAVIORS:
        approved = sum(1 for r in rows if r['behavior']==b and r['status']=='approved')
        pending = sum(1 for r in rows if r['behavior']==b and r['status']=='pending')
        emoji = '✅' if approved >= TARGET_PER_BEHAVIOR else '⏳'
        lines.append(f'{emoji} {b:20s} approved={approved:>2d}/{TARGET_PER_BEHAVIOR}  pending={pending}')
    return '\n'.join(lines)


def parse_hms(t):
    parts = t.split(':')
    if len(parts) == 3:
        h, m, s = parts
    elif len(parts) == 2:
        h = '0'; m, s = parts
    else:
        return 0
    return int(h)*3600 + int(m)*60 + float(s)


def safe_id(s):
    return re.sub(r'[^A-Za-z0-9]+', '_', s)[-40:].strip('_')


def ffmpeg_clip(source, ts_start, ts_end, output):
    cmd = [
        'ffmpeg', '-y', '-loglevel', 'error',
        '-ss', ts_start, '-to', ts_end,
        '-i', source, '-c', 'copy', output,
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        # fallback: re-encode (czasem -c copy nie działa przy keyframe boundary)
        cmd[-3:] = ['-c:v', 'libx264', '-c:a', 'aac', output]
        r = subprocess.run(cmd, capture_output=True, text=True)
    return r.returncode == 0, (r.stderr or '')[-500:]


def load_current(behavior):
    """Zwraca dane bieżącej propozycji do UI."""
    rows, pending = get_pending_for(behavior)
    if not pending:
        return (None, '', '', '', f'Brak pending propozycji dla {behavior}',
                fmt_status_summary())
    STATE['behavior'] = behavior
    STATE['idx'] = pending[0][0]
    r = pending[0][1]
    src = r['source_video']
    label = f"[{r['behavior']}] match='{r['keyword_match']}'  start={r['timestamp_start']}  end={r['timestamp_end']}"
    info = f"Source: {Path(src).name}\n\nContext:\n{r['context']}"
    return src, r['timestamp_start'], r['timestamp_end'], label, info, fmt_status_summary()


def action(decision, behavior, ts_start_in, ts_end_in):
    rows = reload_suggestions()
    if STATE['idx'] >= len(rows):
        return load_current(behavior)
    r = rows[STATE['idx']]

    if decision == 'approve':
        # zaktualizuj timestamps jeśli user zmienił
        ts_s = ts_start_in or r['timestamp_start']
        ts_e = ts_end_in or r['timestamp_end']
        # nazwa output
        idx_for_b = sum(1 for x in rows if x['behavior']==r['behavior'] and x['status']=='approved') + 1
        src_id = safe_id(Path(r['source_video']).stem)
        clip = ANCHORED / r['behavior'] / f"{idx_for_b:02d}_{src_id}.mp4"
        ok, err = ffmpeg_clip(r['source_video'], ts_s, ts_e, str(clip))
        if ok:
            r['status'] = 'approved'
            r['clip_path'] = str(clip.relative_to(POC))
            r['timestamp_start'] = ts_s
            r['timestamp_end'] = ts_e
            print(f'✓ APPROVED → {clip.name}')
        else:
            r['status'] = 'error'
            r['clip_path'] = err[:200]
            print(f'✗ ffmpeg error: {err[:200]}')
    elif decision == 'reject':
        r['status'] = 'rejected'
    save_suggestions(rows)
    return load_current(behavior)


with gr.Blocks(title='RHpE Few-shot Anchor Clipping') as demo:
    gr.Markdown('# RHpE Anchor Clipping — Faza 1.5\n_Approve fragments per behavior. Cel: 5–10 klipów per behavior._')
    with gr.Row():
        behavior_dd = gr.Dropdown(BEHAVIORS, value=BEHAVIORS[0], label='Behavior')
        status_md = gr.Code(value=fmt_status_summary(), label='Status', language='shell')
    label_md = gr.Markdown('Wybierz behavior i kliknij Load.')
    info_md = gr.Code(value='', label='Context z napisów', language='markdown')
    with gr.Row():
        video_player = gr.Video(label='Source video (skocz do timestamp w playerze)', autoplay=False, height=400)
    with gr.Row():
        ts_start_tb = gr.Textbox(label='timestamp_start (HH:MM:SS)', value='')
        ts_end_tb = gr.Textbox(label='timestamp_end (HH:MM:SS)', value='')
    with gr.Row():
        load_btn = gr.Button('🔄 Load Next', variant='secondary')
        approve_btn = gr.Button('✅ Approve & Clip', variant='primary')
        reject_btn = gr.Button('❌ Reject', variant='stop')

    load_btn.click(load_current, [behavior_dd], [video_player, ts_start_tb, ts_end_tb, label_md, info_md, status_md])
    behavior_dd.change(load_current, [behavior_dd], [video_player, ts_start_tb, ts_end_tb, label_md, info_md, status_md])
    approve_btn.click(lambda b, s, e: action('approve', b, s, e), [behavior_dd, ts_start_tb, ts_end_tb],
                      [video_player, ts_start_tb, ts_end_tb, label_md, info_md, status_md])
    reject_btn.click(lambda b, s, e: action('reject', b, s, e), [behavior_dd, ts_start_tb, ts_end_tb],
                      [video_player, ts_start_tb, ts_end_tb, label_md, info_md, status_md])

demo.launch(inbrowser=True, share=False, prevent_thread_lock=True)
print('Gradio UI launched. Pracuj w przeglądarce. Zamknij przez `demo.close()` w nowej komórce.')

## 3. Tryb 2 — Evaluation (uruchom po approve ≥5 klipów per behavior)

Wczytuje wszystkie approved klipy z `vendor/rhpe_anchored/<behavior>/`, ekstraktuje V-JEPA-2 embeddings, leave-one-out cosine similarity per behavior.

In [ ]:
# Sprawdź czy mamy minimum 5 klipów per behavior
import glob
ready = True
for b in BEHAVIORS:
    n = len(list((ANCHORED / b).glob('*.mp4')))
    status = '✓' if n >= TARGET_PER_BEHAVIOR else '✗ za mało'
    print(f'  {b:20s} {n:>3d} klipów  {status}')
    if n < TARGET_PER_BEHAVIOR:
        ready = False

if not ready:
    print('\n⚠ Wróć do Tryb 1 (Gradio UI) i zatwierdź więcej klipów.')
else:
    print('\n✓ Wszystkie behaviors mają ≥5 klipów. Uruchom następną komórkę.')

In [ ]:
# V-JEPA-2 embedding extraction (re-use logiki z notebook 02)
import torch, cv2, numpy as np, time
from transformers import VJEPA2Model, AutoVideoProcessor

MODEL_ID = 'facebook/vjepa2-vitl-fpc16-256-ssv2'
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu'))
print('Device:', DEVICE)

t0 = time.time()
processor = AutoVideoProcessor.from_pretrained(MODEL_ID)
model = VJEPA2Model.from_pretrained(MODEL_ID).to(DEVICE).eval()
print(f'V-JEPA-2 loaded w {time.time()-t0:.1f}s')


def read_clip_frames(clip_path, num_frames=16):
    cap = cv2.VideoCapture(str(clip_path))
    n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if n_total < 1:
        cap.release(); return None
    indices = np.linspace(0, n_total-1, num_frames).astype(int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, f = cap.read()
        if not ok: f = frames[-1] if frames else np.zeros((256,256,3), dtype=np.uint8)
        frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    cap.release()
    return np.stack(frames)


@torch.no_grad()
def extract_embedding(clip_path):
    frames = read_clip_frames(clip_path)
    if frames is None: return None
    inputs = processor(videos=list(frames), return_tensors='pt').to(DEVICE)
    out = model(**inputs)
    return out.last_hidden_state.mean(dim=1).squeeze(0).cpu().float().numpy()


embeddings = []  # list of (behavior, clip_path, emb)
t0 = time.time()
for b in BEHAVIORS:
    for clip in sorted((ANCHORED / b).glob('*.mp4')):
        try:
            emb = extract_embedding(clip)
            if emb is not None:
                embeddings.append((b, str(clip), emb))
                print(f'  ✓ {b:20s} {clip.name}')
        except Exception as e:
            print(f'  ✗ {clip.name}: {e}')
print(f'\n{len(embeddings)} embeddingów w {time.time()-t0:.1f}s')

In [ ]:
# Leave-one-out cosine similarity per behavior
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

labels = [e[0] for e in embeddings]
X = np.stack([e[2] for e in embeddings])
print(f'X shape: {X.shape}')

# Per behavior: dla każdego klipu, oblicz mean cosine similarity z pozostałymi
# klipami tego samego behaviora vs mean cos sim z innych. Pred = behavior z highest mean.
results_per_behavior = {}
all_preds = []
all_truths = []
for i, (true_b, _, _) in enumerate(embeddings):
    sims = cosine_similarity(X[i:i+1], X)[0]  # podobieństwo do wszystkich
    sims[i] = -np.inf  # excluding self (leave-one-out)
    # Per behavior mean similarity
    per_b_mean = {}
    for b in BEHAVIORS:
        mask = [j for j, e in enumerate(embeddings) if e[0]==b and j != i]
        if mask:
            per_b_mean[b] = sims[mask].mean()
    pred_b = max(per_b_mean, key=per_b_mean.get)
    all_preds.append(pred_b)
    all_truths.append(true_b)

# Per-behavior accuracy
for b in BEHAVIORS:
    idxs = [i for i, t in enumerate(all_truths) if t == b]
    n = len(idxs)
    if n == 0:
        continue
    correct = sum(1 for i in idxs if all_preds[i] == b)
    acc = correct / n
    results_per_behavior[b] = {'n': n, 'correct': correct, 'accuracy': acc}
    print(f'  {b:20s} acc={acc:.3f}  ({correct}/{n})')

overall_acc = sum(1 for p, t in zip(all_preds, all_truths) if p==t) / len(all_truths)
print(f'\nOverall accuracy: {overall_acc:.3f}  ({sum(1 for p,t in zip(all_preds,all_truths) if p==t)}/{len(all_truths)})')

# Save results
(POC / 'outputs').mkdir(exist_ok=True)
results_summary = {
    'model': MODEL_ID,
    'method': 'leave-one-out cosine similarity',
    'n_total': len(embeddings),
    'overall_accuracy': float(overall_acc),
    'per_behavior': {b: {'n': v['n'], 'correct': v['correct'], 'accuracy': float(v['accuracy'])}
                     for b, v in results_per_behavior.items()},
    'reference': {
        'paper_claim_alves_2025_ear_movement': 0.875,
        'vjepa2_linear_probe_ear_movement_phase1': 0.854,
    },
}
with open(POC / 'outputs/few_shot_validation_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2)
print(f'\nZapisano: outputs/few_shot_validation_results.json')

In [ ]:
# Plot bar chart
fig, ax = plt.subplots(figsize=(9, 4.5))
behaviors_with_data = [b for b in BEHAVIORS if b in results_per_behavior]
accs = [results_per_behavior[b]['accuracy'] for b in behaviors_with_data]
bars = ax.barh(behaviors_with_data, accs, color='#1f77b4')
ax.axvline(0.5, color='red', linestyle=':', alpha=0.5, label='random (random per 5 klas = 0.20)')
ax.axvline(0.20, color='gray', linestyle=':', alpha=0.5, label='random 5-class')
ax.axvline(0.854, color='green', linestyle='--', alpha=0.5, label='V-JEPA-2 ear movement (Faza 1)')
ax.axvline(0.875, color='black', linestyle='-.', alpha=0.5, label='Paper claim Alves 2025')
ax.axvline(0.70, color='orange', linestyle=':', alpha=0.7, label='Gate Faza 1.5: 0.70')
for bar, val in zip(bars, accs):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=10)
ax.set_xlabel('Accuracy (leave-one-out cosine similarity, 5-class)')
ax.set_xlim(0, 1)
ax.set_title(f'Faza 1.5 — V-JEPA-2 few-shot na 5 RHpE behaviors  (N={len(embeddings)} total)')
ax.legend(loc='lower right', fontsize=8)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(POC / 'outputs/few_shot_validation_plot.png', dpi=120)
plt.show()
print(f'Zapisano: outputs/few_shot_validation_plot.png')

# Gate decision
n_passed = sum(1 for b, v in results_per_behavior.items() if v['accuracy'] >= 0.70)
print(f'\n=== GATE Faza 1.5 ===')
print(f'Behaviors z accuracy ≥0.70: {n_passed}/{len(results_per_behavior)}')
if n_passed >= 3:
    print('✓ GATE PASSED — zielone światło dla Faza 2 (zewnętrzny RHpE assessor)')
else:
    print('✗ GATE NOT PASSED — redukcja scope Fazy 2 lub reflection na podejście')

## 4. Notatka decyzyjna

Po uruchomieniu Tryb 2:
- Sprawdź `outputs/few_shot_validation_results.json` — szczegółowe metryki per behavior
- Sprawdź `outputs/few_shot_validation_plot.png` — wizualizacja vs paper claim i V-JEPA-2 ear movement
- Update `poc/GATE.md` punkt 5 (Faza 1.5) z wynikami

**Decyzja Faza 2:**
- ≥3/5 behaviors z accuracy ≥0,70 → GATE passed → zielone Faza 2 (zewnętrzny assessor 1500–3000 PLN, własny dataset)
- <3/5 → redukcja scope Fazy 2 (skup na behaviors które działają) lub reflection na podejście (może DLC pose-derived features lepsze dla niektórych behaviors)